# Install + imports

In [ ]:
!pip -q install timm

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix

import torch
import torch.nn as nn
from torch.optim import Optimizer
from torch.utils.data import DataLoader, ConcatDataset, WeightedRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
import torch.nn.functional as F

import timm
from tqdm.auto import tqdm


import gc
import json
from pathlib import Path

In [2]:
import sys
sys.path.append("/kaggle/input/datasets/thanhkhil/efficientnet-dualscale")
from efficientnet_dualscale_1 import (
    efficientnet_b2_original,
    efficientnet_b2_dual_scale,
    # efficientnet_b2_dual_scale_arcface,
    # efficientnet_b2_arcface,
    # efficientnet_b2_regional,
    # efficientnet_b2_spatial_refine,
    # efficientnet_b2_compact_head,
    # efficientnet_b2_dual_evidence,
    # efficientnet_b2_c4_refine,
    # efficientnet_b2_upg,
    # count_parameters as count_parameters_upg,
    # prototype_separation_loss,
    # embedding_compactness_loss
)

sys.path.append("/kaggle/input/datasets/thanhkhil/efficientnetb2-archi")
from efficientskindis_variants import (
    EfficientSkin_TADA,
    EfficientSkin_BiFPN,
    EfficientSkin_LWT,
    EfficientSkin_Proto,
    EfficientSkinMSDA,
    EfficientSkin_TADA_Proto,
    EfficientSkin_TADA_GeM,
    EfficientSkin_TADA_Proto_GeM,
    EfficientSkinMSDA_Fixed,
    count_parameters as count_parameters_msda,
)


# Global config

In [ ]:
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset_name = "skin31"

skin31_root = Path("/kaggle/input/datasets/kelixo25/31-classes-of-skin-disease/Atlas dan ISIC2019 (31 classes)")
skin31_train_dir = skin31_root / "train"
skin31_test_dir = skin31_root / "test"

ham_root = Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000")
ham_metadata_path = ham_root / "HAM10000_metadata.csv"
ham_part1_dir = ham_root / "HAM10000_images_part_1"
ham_part2_dir = ham_root / "HAM10000_images_part_2"

work_dir = Path("/kaggle/working")
history_dir = work_dir / "histories"
history_dir.mkdir(parents=True, exist_ok=True)
class_names_path = work_dir / f"class_names_{dataset_name}.json"

img_size = 224
batch_size = 32
epochs = 15
lr = 5e-4
weight_decay = 1e-6
num_workers = 0

# Dataset overview

In [ ]:
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split

HAM_DX_TO_NAME = {
    "akiec": "Actinic keratoses and intraepithelial carcinoma",
    "bcc": "Basal cell carcinoma",
    "bkl": "Benign keratosis-like lesions",
    "df": "Dermatofibroma",
    "mel": "Melanoma",
    "nv": "Melanocytic nevi",
    "vasc": "Vascular lesions"
}

def build_ham10000_dataframe(metadata_path, part1_dir, part2_dir):
    df = pd.read_csv(metadata_path).copy()

    def resolve_path(image_id):
        p1 = part1_dir / f"{image_id}.jpg"
        p2 = part2_dir / f"{image_id}.jpg"
        if p1.exists():
            return str(p1)
        if p2.exists():
            return str(p2)
        return None

    df["path"] = df["image_id"].map(resolve_path)
    df = df[df["path"].notna()].reset_index(drop=True)

    dx_order = sorted(df["dx"].unique().tolist())
    class_names = [HAM_DX_TO_NAME.get(x, x) for x in dx_order]
    dx_to_idx = {dx: i for i, dx in enumerate(dx_order)}

    df["target"] = df["dx"].map(dx_to_idx)
    df["class_name"] = df["dx"].map(lambda x: HAM_DX_TO_NAME.get(x, x))

    return df, dx_order, class_names, dx_to_idx

class HAM10000Dataset(Dataset):
    def __init__(self, df, class_names, transform=None, exclude_paths=None):
        self.df = df.copy().reset_index(drop=True)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        if len(exclude_paths) > 0:
            self.df = self.df[~self.df["path"].isin(exclude_paths)].reset_index(drop=True)

        self.transform = transform
        self.classes = class_names
        self.samples = list(zip(self.df["path"].tolist(), self.df["target"].tolist()))
        self.imgs = self.samples
        self.targets = self.df["target"].tolist()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        target = int(row["target"])

        if self.transform is not None:
            image = self.transform(image)

        return image, target


def build_ham10000_split(test_size=0.2, seed=42):
    ham_df, ham_dx_order, ham_class_names, ham_dx_to_idx = build_ham10000_dataframe(
        metadata_path=ham_metadata_path,
        part1_dir=ham_part1_dir,
        part2_dir=ham_part2_dir
    )

    lesion_df = (
        ham_df.groupby("lesion_id", as_index=False)["target"]
        .first()
        .copy()
    )

    train_lesions, test_lesions = train_test_split(
        lesion_df["lesion_id"],
        test_size=test_size,
        random_state=seed,
        stratify=lesion_df["target"]
    )

    train_df = ham_df[ham_df["lesion_id"].isin(train_lesions)].reset_index(drop=True)
    test_df = ham_df[ham_df["lesion_id"].isin(test_lesions)].reset_index(drop=True)

    train_df = train_df.copy()
    test_df = test_df.copy()

    return train_df, test_df, ham_class_names

In [ ]:
def summarize_ham_split(train_df, test_df, class_names):
    train_counts = train_df["target"].value_counts().sort_index()
    test_counts = test_df["target"].value_counts().sort_index()

    summary_df = pd.DataFrame({
        "class_name": class_names,
        "train_count": [int(train_counts.get(i, 0)) for i in range(len(class_names))],
        "test_count": [int(test_counts.get(i, 0)) for i in range(len(class_names))]
    })

    print("Train images:", len(train_df))
    print("Test images:", len(test_df))
    print("Train lesions:", train_df["lesion_id"].nunique())
    print("Test lesions:", test_df["lesion_id"].nunique())

    display(summary_df)

In [ ]:
ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=42)
summarize_ham_split(ham_train_df, ham_test_df, class_names)

In [ ]:
if dataset_name == "skin31":
    base_train = datasets.ImageFolder(skin31_train_dir)
    base_test = datasets.ImageFolder(skin31_test_dir)

    class_names = base_train.classes
    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

elif dataset_name == "ham10000":
    ham_train_df, ham_test_df, class_names = build_ham10000_split(test_size=0.2, seed=seed)

    base_train = HAM10000Dataset(ham_train_df, class_names=class_names, transform=None)
    base_test = HAM10000Dataset(ham_test_df, class_names=class_names, transform=None)

    num_classes = len(class_names)

    train_counts = pd.Series(base_train.targets).value_counts().sort_index()
    test_counts = pd.Series(base_test.targets).value_counts().sort_index()

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

df_counts = pd.DataFrame({
    "class_name": class_names,
    "train_count": train_counts.values,
    "test_count": test_counts.values
})

class_meta = {
    "dataset_name": dataset_name,
    "num_classes": num_classes,
    "class_names": class_names
}

with open(class_names_path, "w", encoding="utf-8") as f:
    json.dump(class_meta, f, ensure_ascii=False, indent=2)

print("Dataset:", dataset_name)
print("Num classes:", num_classes)
print("Train samples:", len(base_train))
print("Test samples:", len(base_test))

display(df_counts)

# Custom transforms

In [ ]:
class FixedCenterZoom:
    def __init__(self, scale=1.2):
        self.scale = scale

    def __call__(self, img):
        w, h = img.size
        nw, nh = int(w * self.scale), int(h * self.scale)
        img = img.resize((nw, nh), Image.BILINEAR)
        left = (nw - w) // 2
        top = (nh - h) // 2
        return img.crop((left, top, left + w, top + h))


class FixedRotate:
    def __init__(self, angle=20):
        self.angle = angle

    def __call__(self, img):
        return TF.rotate(img, self.angle, interpolation=transforms.InterpolationMode.BILINEAR, fill=0)


class FixedBrightness:
    def __init__(self, factor=1.2):
        self.factor = factor

    def __call__(self, img):
        return ImageEnhance.Brightness(img).enhance(self.factor)


class FixedShear:
    def __init__(self, shear=12):
        self.shear = shear

    def __call__(self, img):
        return TF.affine(
            img,
            angle=0,
            translate=[0, 0],
            scale=1.0,
            shear=[self.shear, 0],
            interpolation=transforms.InterpolationMode.BILINEAR,
            fill=0
        )

sample_path, sample_label = base_train.samples[0]
sample_img = Image.open(sample_path).convert("RGB")

preview = {
    "Original": sample_img,
    "Center Zoom": FixedCenterZoom(1.2)(sample_img.copy()),
    "Rotation": FixedRotate(20)(sample_img.copy()),
    "Brightness": FixedBrightness(1.2)(sample_img.copy()),
    "Shear": FixedShear(12)(sample_img.copy()),
    "Vertical Flip": TF.vflip(sample_img.copy()),
    "Horizontal Flip": TF.hflip(sample_img.copy())
}

fig, axes = plt.subplots(1, 7, figsize=(24, 4))

for ax, (name, img) in zip(axes, preview.items()):
    ax.imshow(img)
    ax.set_title(name)
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Sample class:", class_names[sample_label])

# Dataset filtering helper

In [ ]:
class FilteredImageFolder(datasets.ImageFolder):
    def __init__(self, root, transform=None, exclude_paths=None):
        super().__init__(root=root, transform=transform)
        exclude_paths = {str(Path(p)) for p in (exclude_paths or [])}

        filtered_samples = []
        filtered_targets = []

        for path, target in self.samples:
            if str(Path(path)) not in exclude_paths:
                filtered_samples.append((path, target))
                filtered_targets.append(target)

        self.samples = filtered_samples
        self.imgs = filtered_samples
        self.targets = filtered_targets

# Transform definitions

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

to_tensor_norm = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])
orig_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

zoom_tf = transforms.Compose([
    FixedCenterZoom(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

rot_tf = transforms.Compose([
    FixedRotate(20),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

bright_tf = transforms.Compose([
    FixedBrightness(1.2),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

shear_tf = transforms.Compose([
    FixedShear(12),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

vflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.vflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

hflip_tf = transforms.Compose([
    transforms.Lambda(lambda x: TF.hflip(x)),
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

# Training utilities

In [ ]:
def rand_bbox(size, lam):
    H = size[2]
    W = size[3]

    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)

    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    return x1, y1, x2, y2


def apply_cutmix(images, labels, alpha=1.0):
    if alpha <= 0:
        return images, labels, labels, 1.0

    lam = np.random.beta(alpha, alpha)
    rand_index = torch.randperm(images.size(0), device=images.device)

    labels_a = labels
    labels_b = labels[rand_index]

    x1, y1, x2, y2 = rand_bbox(images.size(), lam)
    images = images.clone()
    images[:, :, y1:y2, x1:x2] = images[rand_index, :, y1:y2, x1:x2]

    lam = 1.0 - ((x2 - x1) * (y2 - y1) / (images.size(2) * images.size(3)))

    return images, labels_a, labels_b, lam

In [ ]:
class SAM(Optimizer):
    def __init__(self, params, base_optimizer_cls, rho=0.05, adaptive=False, **kwargs):
        if rho < 0.0:
            raise ValueError(f"Invalid rho, should be non-negative: {rho}")

        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super().__init__(params, defaults)

        self.base_optimizer = base_optimizer_cls(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)
        self.rho = rho
        self.adaptive = adaptive

    @torch.no_grad()
    def _grad_norm(self):
        device = self.param_groups[0]["params"][0].device
        norms = []

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                if group.get("adaptive", self.adaptive):
                    norms.append((torch.abs(p) * p.grad).norm(p=2).to(device))
                else:
                    norms.append(p.grad.norm(p=2).to(device))

        if len(norms) == 0:
            return torch.tensor(0.0, device=device)

        return torch.norm(torch.stack(norms), p=2)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()

        if grad_norm.item() == 0.0:
            if zero_grad:
                self.zero_grad(set_to_none=True)
            return

        for group in self.param_groups:
            scale = group.get("rho", self.rho) / (grad_norm + 1e-12)

            for p in group["params"]:
                if p.grad is None:
                    continue

                if group.get("adaptive", self.adaptive):
                    e_w = torch.pow(p, 2) * p.grad * scale
                else:
                    e_w = p.grad * scale

                p.add_(e_w)
                self.state[p]["e_w"] = e_w

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                e_w = self.state[p].get("e_w", None)
                if e_w is not None:
                    p.sub_(e_w)

        if zero_grad:
            self.zero_grad(set_to_none=True)

    @torch.no_grad()
    def step(self, closure=None):
        raise NotImplementedError("Use first_step and second_step with SAM.")

    def zero_grad(self, set_to_none=True):
        self.base_optimizer.zero_grad(set_to_none=set_to_none)

In [ ]:
def kd_kl_loss(student_logits, teacher_logits, temperature=4.0):
    log_p = F.log_softmax(student_logits / temperature, dim=1)
    q = F.softmax(teacher_logits / temperature, dim=1)
    return F.kl_div(log_p, q, reduction="batchmean") * (temperature ** 2)

In [ ]:
def prepare_batch(images, labels, use_cutmix=False, cutmix_alpha=1.0):
    if use_cutmix:
        images, labels_a, labels_b, lam = apply_cutmix(images, labels, alpha=cutmix_alpha)
        return images, labels_a, labels_b, lam
    return images, labels, labels, 1.0


def forward_loss(
    model,
    images,
    labels,
    criterion,
    labels_a=None,
    labels_b=None,
    lam=1.0,
    use_amp=True,
    aux_weight=0.2,
    coarse_weight=0.0,
    fine_to_coarse_tensor=None,
    compact_weight=0.0,
    sep_weight=0.0,
    # ── MỚI: KD args ───────────────────────────────────────────
    teacher_model=None,
    kd_alpha=0.0,
    kd_temperature=4.0,
):
    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    uses_margin_head = getattr(model, "uses_margin_head", False)
    uses_aux_head = getattr(model, "uses_aux_head", False)
    uses_coarse_head = getattr(model, "uses_coarse_head", False)
 
    with torch.amp.autocast(amp_device, enabled=use_amp and torch.cuda.is_available()):
        if uses_margin_head:
            if labels_a is not None and labels_b is not None and lam != 1.0:
                outputs_a = model(images, target=labels_a)
                outputs_b = model(images, target=labels_b)
                loss = lam * criterion(outputs_a, labels_a) + (1.0 - lam) * criterion(outputs_b, labels_b)
                outputs = lam * outputs_a + (1.0 - lam) * outputs_b
            else:
                outputs = model(images, target=labels)
                loss = criterion(outputs, labels)
 
        elif uses_aux_head:
            main_outputs, aux_outputs = model(images, return_aux=True)
            if labels_a is not None and labels_b is not None and lam != 1.0:
                main_loss = lam * criterion(main_outputs, labels_a) + (1.0 - lam) * criterion(main_outputs, labels_b)
                aux_loss = lam * criterion(aux_outputs, labels_a) + (1.0 - lam) * criterion(aux_outputs, labels_b)
            else:
                main_loss = criterion(main_outputs, labels)
                aux_loss = criterion(aux_outputs, labels)
            loss = main_loss + aux_weight * aux_loss
            outputs = main_outputs
 
        elif uses_coarse_head:
            parts = model(images, return_parts=True)
            outputs = parts["logits"]
            coarse_outputs = parts["coarse_logits"]
            if labels_a is not None and labels_b is not None and lam != 1.0:
                fine_loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
                if coarse_weight > 0.0 and fine_to_coarse_tensor is not None:
                    coarse_a = fine_to_coarse_tensor[labels_a]; coarse_b = fine_to_coarse_tensor[labels_b]
                    coarse_loss = lam * F.cross_entropy(coarse_outputs, coarse_a) + (1.0 - lam) * F.cross_entropy(coarse_outputs, coarse_b)
                else:
                    coarse_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
                if compact_weight > 0.0:
                    compact_a = embedding_compactness_loss(model, parts, labels_a)
                    compact_b = embedding_compactness_loss(model, parts, labels_b)
                    compact_loss = lam * compact_a + (1.0 - lam) * compact_b
                else:
                    compact_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            else:
                fine_loss = criterion(outputs, labels)
                if coarse_weight > 0.0 and fine_to_coarse_tensor is not None:
                    coarse_labels = fine_to_coarse_tensor[labels]
                    coarse_loss = F.cross_entropy(coarse_outputs, coarse_labels)
                else:
                    coarse_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
                if compact_weight > 0.0:
                    compact_loss = embedding_compactness_loss(model, parts, labels)
                else:
                    compact_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            if sep_weight > 0.0:
                sep_loss = prototype_separation_loss(model)
            else:
                sep_loss = torch.zeros((), device=outputs.device, dtype=outputs.dtype)
            loss = fine_loss + coarse_weight * coarse_loss + compact_weight * compact_loss + sep_weight * sep_loss
 
        elif getattr(model, 'uses_msda_head', False):
            if labels_a is not None and labels_b is not None and lam != 1.0:
                outputs, proto_loss = model(images, labels_a)
                main_loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                outputs, proto_loss = model(images, labels)
                main_loss = criterion(outputs, labels)
            loss = main_loss + aux_weight * proto_loss
 
        else:
            outputs = model(images)
            if labels_a is not None and labels_b is not None and lam != 1.0:
                loss = lam * criterion(outputs, labels_a) + (1.0 - lam) * criterion(outputs, labels_b)
            else:
                loss = criterion(outputs, labels)
 
        # ── MỚI: Logit Knowledge Distillation ──────────────────────────────
        # L = (1-alpha)*CE + alpha*KL(student||teacher)
        if teacher_model is not None and kd_alpha > 0.0:
            with torch.no_grad():
                t_out = teacher_model(images)              # teacher logits (frozen)
            kd = kd_kl_loss(outputs, t_out, temperature=kd_temperature)
            loss = (1.0 - kd_alpha) * loss + kd_alpha * kd
 
    return outputs, loss


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    aux_weight=0.2,
    coarse_weight=0.0,
    fine_to_coarse_tensor=None,
    compact_weight=0.0,
    sep_weight=0.0,
    teacher_model=None,
    kd_alpha=0.0,
    kd_temperature=4.0,
):
    model.train()

    running_loss = 0.0
    correct = 0.0
    total = 0

    pbar = tqdm(loader, leave=False)

    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        images, labels_a, labels_b, lam = prepare_batch(
            images=images,
            labels=labels,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha
        )

        optimizer.zero_grad(set_to_none=True)

        if not use_sam:
            outputs, loss = forward_loss(
                model=model,
                images=images,
                labels=labels,
                criterion=criterion,
                labels_a=labels_a if use_cutmix else None,
                labels_b=labels_b if use_cutmix else None,
                lam=lam,
                use_amp=True,
                aux_weight=aux_weight,
                coarse_weight=coarse_weight,
                fine_to_coarse_tensor=fine_to_coarse_tensor,
                compact_weight=compact_weight,
                sep_weight=sep_weight,
                teacher_model=teacher_model,
                kd_alpha=kd_alpha,
                kd_temperature=kd_temperature,
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss detected: {loss.item()}")

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            if use_cutmix:
                correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
            else:
                correct += (preds == labels).sum().item()

            total += labels.size(0)
            pbar.set_postfix(loss=running_loss / total, acc=correct / total)
            continue

        outputs_1, loss_1 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=fine_to_coarse_tensor,
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )

        if not torch.isfinite(loss_1):
            raise RuntimeError(f"Non-finite loss detected at SAM first pass: {loss_1.item()}")

        loss_1.backward()
        optimizer.first_step(zero_grad=True)

        _, loss_2 = forward_loss(
            model=model,
            images=images,
            labels=labels,
            criterion=criterion,
            labels_a=labels_a if use_cutmix else None,
            labels_b=labels_b if use_cutmix else None,
            lam=lam,
            use_amp=False,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=fine_to_coarse_tensor,
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )

        if not torch.isfinite(loss_2):
            raise RuntimeError(f"Non-finite loss detected at SAM second pass: {loss_2.item()}")

        loss_2.backward()
        optimizer.second_step(zero_grad=False)
        optimizer.base_optimizer.step()
        optimizer.zero_grad(set_to_none=True)

        running_loss += loss_1.item() * images.size(0)
        preds = outputs_1.argmax(dim=1)

        if use_cutmix:
            correct += lam * (preds == labels_a).sum().item() + (1.0 - lam) * (preds == labels_b).sum().item()
        else:
            correct += (preds == labels).sum().item()

        total += labels.size(0)
        pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"

    with torch.no_grad():
        pbar = tqdm(loader, leave=False)

        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(amp_device, enabled=torch.cuda.is_available()):
                if getattr(model, 'uses_msda_head', False):
                    outputs, _ = model(images)  # discard proto_loss at eval
                else:
                    outputs = model(images)
                loss = criterion(outputs, labels)

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite eval loss detected: {loss.item()}")

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

            pbar.set_postfix(loss=running_loss / total, acc=correct / total)

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

In [ ]:
coarse_groups = {
    "malignant_premalignant": [
        "Actinic keratosis",
        "Basal Cell Carcinoma",
        "Melanoma",
        "Mycosis Fungoides",
        "Porokeratosis Actinic",
        "Squamous cell carcinoma"
    ],
    "benign_pigmented_keratotic": [
        "Nevus",
        "Pigmented Benign Keratosis",
        "Seborrheic keratosis"
    ],
    "infectious_bacterial_viral": [
        "Herpes Simplex",
        "Impetigo",
        "Leprosy Borderline",
        "Leprosy Lepromatous",
        "Leprosy Tuberculoid",
        "Molluscum Contagiosum"
    ],
    "infectious_parasitic_fungal": [
        "Larva Migrans",
        "Pediculosis Capitis",
        "Tinea Corporis",
        "Tinea Nigra",
        "Tungiasis"
    ],
    "inflammatory_autoimmune_papulosquamous": [
        "Lichen Planus",
        "Lupus Erythematosus Chronicus Discoides",
        "Pityriasis Rosea",
        "Psoriasis",
        "Papilomatosis Confluentes And Reticulate"
    ],
    "genodermatosis_blistering": [
        "Darier_s Disease",
        "Epidermolysis Bullosa Pruriginosa",
        "Hailey-Hailey Disease"
    ],
    "benign_fibrous_neural_vascular": [
        "Dermatofibroma",
        "Neurofibromatosis",
        "Vascular lesion"
    ]
}

coarse_name_to_idx = {name: i for i, name in enumerate(coarse_groups.keys())}
fine_to_coarse = {}

for coarse_name, fine_names in coarse_groups.items():
    for fine_name in fine_names:
        fine_to_coarse[class_names.index(fine_name)] = coarse_name_to_idx[coarse_name]

missing = [name for i, name in enumerate(class_names) if i not in fine_to_coarse]
print("Missing:", missing)

fine_to_coarse_list = [fine_to_coarse[i] for i in range(len(class_names))]

fine_to_coarse_tensor = torch.tensor(
    fine_to_coarse_list,
    dtype=torch.long,
    device=device
)

num_coarse_classes = len(coarse_groups)

for i, name in enumerate(class_names):
    print(i, name, "->", list(coarse_groups.keys())[fine_to_coarse_list[i]])

print("num_coarse_classes:", num_coarse_classes)
print("fine_to_coarse_list:", fine_to_coarse_list)
print("fine_to_coarse_tensor:", fine_to_coarse_tensor)

# Model/optimizer/scheduler utilities

In [ ]:
def build_model(
    model_name="efficientnet_b2",
    model_variant="original",
    drop_rate=0.0,
    pretrained=True,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=24.0,
    arcface_m=0.15,
    region_grid=2,
    use_gem=True,
    attn_hidden=128,
    refine_kernel_size=7,
    use_cosine_classifier=False,
    cosine_s=24.0,
    top_k=3,
    score_hidden=128,
    mamba_dim=96,
    d_state=16,
    d_conv=3,
    expand=2,
    init_alpha=0.1,
    use_external_mamba=False,
    prototype_scale=12.0,
    classifier_scale=1.0,
    uncertainty_hidden=128,
    num_coarse_classes=0,
    fine_to_coarse=None,
    residual_scale=0.5,
    coarse_scale=12.0,
    gate_temperature=1.0,
    # ── MSDA params ─────────────────────
    bifpn_dim=128,
    proto_temperature=0.07,
    msda_transformer_depth=2,
    msda_freeze_stages=2,
):
    if model_name != "efficientnet_b2":
        model = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=num_classes,
            drop_rate=drop_rate
        )
    else:
        if model_variant == "original":
            model = efficientnet_b2_original(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "dual_scale":
            model = efficientnet_b2_dual_scale(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels
            )
        elif model_variant == "dual_scale_arcface":
            model = efficientnet_b2_dual_scale_arcface(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fusion_channels=fusion_channels,
                embedding_dim=embedding_dim,
                arcface_s=arcface_s,
                arcface_m=arcface_m
            )
        elif model_variant == "arcface":
            model = efficientnet_b2_arcface(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                arcface_s=arcface_s,
                arcface_m=arcface_m
            )
        elif model_variant == "regional":
            model = efficientnet_b2_regional(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                region_grid=region_grid,
                use_gem=use_gem,
                attn_hidden=attn_hidden
            )
        elif model_variant == "spatial_refine":
            model = efficientnet_b2_spatial_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                refine_kernel_size=refine_kernel_size
            )
        elif model_variant == "compact_head":
            model = efficientnet_b2_compact_head(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                use_gem=use_gem,
                cosine_s=cosine_s,
                use_cosine_classifier=use_cosine_classifier
            )
        elif model_variant == "dual_evidence":
            model = efficientnet_b2_dual_evidence(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                region_grid=region_grid,
                top_k=top_k,
                use_gem=use_gem,
                score_hidden=score_hidden
            )
        elif model_variant == "c4_refine":
            model = efficientnet_b2_c4_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate
            )
        elif model_variant == "mamba_refine":
            model = efficientnet_b2_mamba_refine(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                mamba_dim=mamba_dim,
                d_state=d_state,
                d_conv=d_conv,
                expand=expand,
                init_alpha=init_alpha,
                use_external_mamba=use_external_mamba
            )
        elif model_variant == "upg":
            model = efficientnet_b2_upg(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                embedding_dim=embedding_dim,
                prototype_scale=prototype_scale,
                classifier_scale=classifier_scale,
                uncertainty_hidden=uncertainty_hidden,
                use_gem=use_gem,
                num_coarse_classes=num_coarse_classes,
                fine_to_coarse=fine_to_coarse,
                residual_scale=residual_scale,
                coarse_scale=coarse_scale,
                gate_temperature=gate_temperature
            )
        elif model_variant == 'msda_tada':
            model = EfficientSkin_TADA(
                num_classes=num_classes, drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant == 'msda_bifpn':
            model = EfficientSkin_BiFPN(
                num_classes=num_classes, D=bifpn_dim,
                drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant == 'msda_lwt':
            model = EfficientSkin_LWT(
                num_classes=num_classes, transformer_dim=bifpn_dim,
                drop_rate=drop_rate, pretrained=pretrained)
        elif model_variant in ('msda_proto', 'msda_full'):
            if model_variant == 'msda_proto':
                model = EfficientSkin_Proto(
                    num_classes=num_classes, drop_rate=drop_rate,
                    proto_temperature=proto_temperature, pretrained=pretrained)
            else:
                model = EfficientSkinMSDA(
                    num_classes=num_classes, D=bifpn_dim,
                    transformer_depth=msda_transformer_depth,
                    drop_rate=drop_rate, proto_temperature=proto_temperature,
                    pretrained=pretrained, freeze_backbone_stages=msda_freeze_stages)
            model.uses_msda_head = True  # flag for forward_loss / evaluate
        elif model_variant == "msda_tada_proto":
            model = EfficientSkin_TADA_Proto(
                num_classes=num_classes, drop_rate=drop_rate,
                proto_temperature=proto_temperature, pretrained=pretrained)
            model.uses_msda_head = True

        elif model_variant == "msda_tada_gem":
            model = EfficientSkin_TADA_GeM(
                num_classes=num_classes, drop_rate=drop_rate, pretrained=pretrained)

        elif model_variant == "msda_tada_proto_gem":
            model = EfficientSkin_TADA_Proto_GeM(
                num_classes=num_classes, drop_rate=drop_rate,
                proto_temperature=proto_temperature, pretrained=pretrained)
            model.uses_msda_head = True

        elif model_variant == "msda_full_fixed":
            model = EfficientSkinMSDA_Fixed(
                num_classes=num_classes, D=bifpn_dim,
                transformer_depth=msda_transformer_depth,
                drop_rate=drop_rate, proto_temperature=proto_temperature,
                pretrained=pretrained, freeze_backbone_stages=msda_freeze_stages)
            model.uses_msda_head = True
        elif model_variant == "wagf":
            model = WAGFFusion_EfficientNetB2(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                learnable_fusion=False,)
        elif model_variant == "eca":
            model = AttnEfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                attn="eca",
            )
 
        elif model_variant == "cbam":
            model = AttnEfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                attn="cbam",
            )
        elif model_variant == "mssf":
            model = MSSF_EfficientNet(
                num_classes=num_classes,
                pretrained=pretrained,
                drop_rate=drop_rate,
                fuse_dim=256,
                out_indices=(2, 3, 4),   # 28×28, 14×14, 7×7
            )
        else:
            raise ValueError(f"Unsupported model_variant: {model_variant}")

    return model.to(device)


def build_optimizer(model, optimizer_name, lr_value=None, use_sam=False, sam_rho=0.05, sam_adaptive=False):
    actual_lr = lr if lr_value is None else lr_value

    if not use_sam:
        if optimizer_name == "adam":
            return torch.optim.Adam(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        if optimizer_name == "adamw":
            return torch.optim.AdamW(model.parameters(), lr=actual_lr, weight_decay=weight_decay)
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    if optimizer_name == "adam":
        return SAM(
            model.parameters(),
            torch.optim.Adam,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    if optimizer_name == "adamw":
        return SAM(
            model.parameters(),
            torch.optim.AdamW,
            rho=sam_rho,
            adaptive=sam_adaptive,
            lr=actual_lr,
            weight_decay=weight_decay
        )

    raise ValueError(f"Unsupported optimizer: {optimizer_name}")


def build_scheduler(optimizer, scheduler_name):
    target_optimizer = optimizer.base_optimizer if hasattr(optimizer, "base_optimizer") else optimizer

    if scheduler_name == "none":
        return None
    if scheduler_name == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(target_optimizer, T_max=epochs)
    raise ValueError(f"Unsupported scheduler: {scheduler_name}")


def slugify_exp_name(exp_name):
    return exp_name.replace(" ", "_").replace("+", "plus").lower()


def save_history_files(exp_name, history):
    exp_slug = slugify_exp_name(exp_name)
    hist_df = pd.DataFrame(history)
    csv_path = history_dir / f"{exp_slug}_history.csv"
    json_path = history_dir / f"{exp_slug}_history.json"
    hist_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    return csv_path, json_path


def save_summary_files(all_results):
    summary_df = pd.DataFrame(all_results)
    csv_path = work_dir / "experiment_summary.csv"
    json_path = work_dir / "experiment_summary.json"
    summary_df.to_csv(csv_path, index=False)
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    return summary_df, csv_path, json_path


def run_experiment(
    exp_name,
    optimizer_name,
    scheduler_name,
    model_name="efficientnet_b2",
    model_variant="original",
    lr_value=None,
    label_smoothing=0.0,
    use_cutmix=False,
    cutmix_alpha=1.0,
    use_sam=False,
    sam_rho=0.05,
    sam_adaptive=False,
    drop_rate=0.0,
    fusion_channels=512,
    embedding_dim=256,
    arcface_s=30.0,
    arcface_m=0.3,
    aux_weight=0.2,
    region_grid=2,
    use_gem=True,
    attn_hidden=128,
    refine_kernel_size=7,
    use_cosine_classifier=False,
    cosine_s=24.0,
    top_k=3,
    score_hidden=128,
    mamba_dim=96,
    d_state=16,
    d_conv=3,
    expand=2,
    init_alpha=0.1,
    use_external_mamba=False,
    prototype_scale=12.0,
    classifier_scale=1.0,
    uncertainty_hidden=128,
    use_kd=False,
    kd_teacher_path=None,
    kd_alpha=0.3,
    kd_temperature=4.0,
    kd_teacher_variant="dual_scale",
    kd_teacher_fusion_channels=512,
    num_coarse_classes=0,
    fine_to_coarse=None,
    residual_scale=0.5,
    coarse_scale=12.0,
    coarse_weight=0.0,
    compact_weight=0.0,
    sep_weight=0.0,
    gate_temperature=1.0,
    run_diagnostic=True,
    baseline_checkpoint=None,
    # ── MSDA ─────────────────────────────
    bifpn_dim=128,
    proto_temperature=0.07,
    msda_transformer_depth=2,
    msda_freeze_stages=2,
):
    actual_lr = lr if lr_value is None else lr_value

    print("=" * 80)
    print(f"Running: {exp_name} | model={model_name} | variant={model_variant} | lr={actual_lr} | sam={use_sam}")
    print("=" * 80)

    exp_slug = slugify_exp_name(exp_name)
    save_path = work_dir / f"{exp_slug}_best.pth"

    model = build_model(
        model_name=model_name,
        model_variant=model_variant,
        drop_rate=drop_rate,
        pretrained=True,
        fusion_channels=fusion_channels,
        embedding_dim=embedding_dim,
        arcface_s=arcface_s,
        arcface_m=arcface_m,
        region_grid=region_grid,
        use_gem=use_gem,
        attn_hidden=attn_hidden,
        refine_kernel_size=refine_kernel_size,
        use_cosine_classifier=use_cosine_classifier,
        cosine_s=cosine_s,
        top_k=top_k,
        score_hidden=score_hidden,
        mamba_dim=mamba_dim,
        d_state=d_state,
        d_conv=d_conv,
        expand=expand,
        init_alpha=init_alpha,
        use_external_mamba=use_external_mamba,
        prototype_scale=prototype_scale,
        classifier_scale=classifier_scale,
        uncertainty_hidden=uncertainty_hidden,
        num_coarse_classes=num_coarse_classes,
        fine_to_coarse=fine_to_coarse,
        residual_scale=residual_scale,
        coarse_scale=coarse_scale,
        gate_temperature=gate_temperature,
        bifpn_dim=bifpn_dim,
        proto_temperature=proto_temperature,
        msda_transformer_depth=msda_transformer_depth,
        msda_freeze_stages=msda_freeze_stages,
    )

    teacher_model = None
    if use_kd and kd_teacher_path is not None:
        state = torch.load(kd_teacher_path, map_location=device)
        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]
        elif isinstance(state, dict) and "model" in state:
            state = state["model"]
    
        teacher_model = efficientnet_b2_dual_scale(
            num_classes=num_classes,
            pretrained=False,
            fusion_channels=kd_teacher_fusion_channels,   # 512
        )
        missing, unexpected = teacher_model.load_state_dict(state, strict=False)
        print(f"[KD] Teacher loaded. missing={len(missing)} unexpected={len(unexpected)}")
        teacher_model = teacher_model.to(device).eval()
        for p in teacher_model.parameters():
            p.requires_grad = False

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = build_optimizer(
        model=model,
        optimizer_name=optimizer_name,
        lr_value=lr_value,
        use_sam=use_sam,
        sam_rho=sam_rho,
        sam_adaptive=sam_adaptive
    )
    scheduler = build_scheduler(optimizer, scheduler_name)

    amp_device = "cuda" if torch.cuda.is_available() else "cpu"
    scaler = torch.amp.GradScaler(amp_device, enabled=torch.cuda.is_available())

    best_acc = 0.0
    best_epoch = 0
    history = []

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
            use_cutmix=use_cutmix,
            cutmix_alpha=cutmix_alpha,
            use_sam=use_sam,
            aux_weight=aux_weight,
            coarse_weight=coarse_weight,
            fine_to_coarse_tensor=fine_to_coarse_tensor if coarse_weight > 0.0 or compact_weight > 0.0 else None,
            compact_weight=compact_weight,
            sep_weight=sep_weight,
            teacher_model=teacher_model,
            kd_alpha=kd_alpha,
            kd_temperature=kd_temperature,
        )

        val_loss, val_acc, y_true, y_pred = evaluate(
            model=model,
            loader=test_loader,
            criterion=criterion,
            device=device
        )

        if scheduler is not None:
            scheduler.step()

        row = {
            "experiment": exp_name,
            "model_name": model_name,
            "model_variant": model_variant,
            "optimizer": optimizer_name,
            "scheduler": scheduler_name,
            "lr": float(actual_lr),
            "label_smoothing": float(label_smoothing),
            "use_cutmix": bool(use_cutmix),
            "cutmix_alpha": float(cutmix_alpha),
            "use_sam": bool(use_sam),
            "sam_rho": float(sam_rho),
            "sam_adaptive": bool(sam_adaptive),
            "drop_rate": float(drop_rate),
            "fusion_channels": int(fusion_channels),
            "embedding_dim": int(embedding_dim),
            "arcface_s": float(arcface_s),
            "arcface_m": float(arcface_m),
            "aux_weight": float(aux_weight),
            "region_grid": int(region_grid),
            "top_k": int(top_k),
            "use_gem": bool(use_gem),
            "attn_hidden": int(attn_hidden),
            "score_hidden": int(score_hidden),
            "refine_kernel_size": int(refine_kernel_size),
            "use_cosine_classifier": bool(use_cosine_classifier),
            "cosine_s": float(cosine_s),
            "mamba_dim": int(mamba_dim),
            "d_state": int(d_state),
            "d_conv": int(d_conv),
            "expand": int(expand),
            "init_alpha": float(init_alpha),
            "use_external_mamba": bool(use_external_mamba),
            "prototype_scale": float(prototype_scale),
            "classifier_scale": float(classifier_scale),
            "uncertainty_hidden": int(uncertainty_hidden),
            "use_kd": bool(use_kd),
            "kd_teacher_path": str(kd_teacher_path) if kd_teacher_path is not None else None,
            "kd_alpha": float(kd_alpha),
            "kd_temperature": float(kd_temperature),
            "epoch": epoch,
            "train_loss": float(train_loss),
            "train_acc": float(train_acc),
            "test_loss": float(val_loss),
            "test_acc": float(val_acc),
            "num_coarse_classes": int(num_coarse_classes) if num_coarse_classes is not None else 0,
            "residual_scale": float(residual_scale),
            "coarse_scale": float(coarse_scale),
            "coarse_weight": float(coarse_weight),
            "compact_weight": float(compact_weight),
            "sep_weight": float(sep_weight),
            "gate_temperature": float(gate_temperature),
        }
        history.append(row)

        save_history_files(exp_name, history)

        print(
            f"[{exp_name}] "
            f"Epoch {epoch:02d}/{epochs} | "
            f"lr={actual_lr:.6f} | "
            f"train_loss={train_loss:.4f} | "
            f"train_acc={train_acc:.4f} | "
            f"test_loss={val_loss:.4f} | "
            f"test_acc={val_acc:.4f}"
        )

        if val_acc > best_acc:
            best_acc = float(val_acc)
            best_epoch = epoch
            torch.save(model.state_dict(), save_path)

    print(f"[{exp_name}] Best test accuracy: {best_acc:.4f} at epoch {best_epoch}")
    print(f"[{exp_name}] Saved best model to: {save_path}")

    del model, criterion, optimizer, scheduler, scaler

    if teacher_model is not None:
        del teacher_model
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    diagnostic_result = None
    
    if run_diagnostic and save_path.exists():
        model_kwargs_for_diag = {}
    
        if model_variant == "upg":
            model_kwargs_for_diag = {
                "embedding_dim": embedding_dim,
                "prototype_scale": prototype_scale,
                "classifier_scale": classifier_scale,
                "uncertainty_hidden": uncertainty_hidden,
                "use_gem": use_gem,
                "num_coarse_classes": num_coarse_classes,
                "fine_to_coarse": fine_to_coarse,
                "residual_scale": residual_scale,
                "coarse_scale": coarse_scale,
                "gate_temperature": gate_temperature
            }
    
        diagnostic_result = evaluate_checkpoint_after_training(
            checkpoint_path=save_path,
            exp_name=exp_name,
            model_name=model_name,
            model_variant=model_variant,
            model_kwargs=model_kwargs_for_diag,
            baseline_checkpoint=baseline_checkpoint
        )

    return {
        "experiment": exp_name,
        "model_name": model_name,
        "model_variant": model_variant,
        "optimizer": optimizer_name,
        "scheduler": scheduler_name,
        "lr": float(actual_lr),
        "label_smoothing": float(label_smoothing),
        "use_cutmix": bool(use_cutmix),
        "cutmix_alpha": float(cutmix_alpha),
        "use_sam": bool(use_sam),
        "sam_rho": float(sam_rho),
        "sam_adaptive": bool(sam_adaptive),
        "drop_rate": float(drop_rate),
        "fusion_channels": int(fusion_channels),
        "embedding_dim": int(embedding_dim),
        "arcface_s": float(arcface_s),
        "arcface_m": float(arcface_m),
        "aux_weight": float(aux_weight),
        "region_grid": int(region_grid),
        "top_k": int(top_k),
        "use_gem": bool(use_gem),
        "attn_hidden": int(attn_hidden),
        "score_hidden": int(score_hidden),
        "refine_kernel_size": int(refine_kernel_size),
        "use_cosine_classifier": bool(use_cosine_classifier),
        "cosine_s": float(cosine_s),
        "mamba_dim": int(mamba_dim),
        "d_state": int(d_state),
        "d_conv": int(d_conv),
        "expand": int(expand),
        "init_alpha": float(init_alpha),
        "use_external_mamba": bool(use_external_mamba),
        "prototype_scale": float(prototype_scale),
        "classifier_scale": float(classifier_scale),
        "uncertainty_hidden": int(uncertainty_hidden),
        "use_kd": bool(use_kd),
        "kd_teacher_path": str(kd_teacher_path) if kd_teacher_path is not None else None,
        "kd_alpha": float(kd_alpha),
        "kd_temperature": float(kd_temperature),
        "num_coarse_classes": int(num_coarse_classes) if num_coarse_classes is not None else 0,
        "residual_scale": float(residual_scale),
        "coarse_scale": float(coarse_scale),
        "coarse_weight": float(coarse_weight),
        "compact_weight": float(compact_weight),
        "sep_weight": float(sep_weight),
        "diagnostic_json": str(work_dir / f"{exp_slug}_diagnostic.json") if diagnostic_result is not None else None,
        "run_diagnostic": bool(run_diagnostic),
        "baseline_checkpoint": str(baseline_checkpoint) if baseline_checkpoint is not None else None,
        "gate_temperature": float(gate_temperature),
        "best_acc": best_acc,
        "best_epoch": best_epoch,
        "save_path": str(save_path),
        "status": "ok"
    }

In [ ]:
# # =====================================================================
# # Q1 Upgrade — B2 + GeM + Sub-Center ArcFace + Auxiliary Self-KD
# # Drop-in extension. Existing functions in cells 21 & 24 are monkey-patched.
# # =====================================================================
# import math
# from typing import Dict, Optional


# # ---------------------------------------------------------------------
# # Building blocks
# # ---------------------------------------------------------------------
# class GeMPool(nn.Module):
#     """Generalized Mean Pooling. p=1 → avg, p→∞ → max. Learnable p."""
#     def __init__(self, p: float = 3.0, eps: float = 1e-6, learnable: bool = True):
#         super().__init__()
#         if learnable:
#             self.p = nn.Parameter(torch.ones(1) * p)
#         else:
#             self.register_buffer('p', torch.ones(1) * p)
#         self.eps = eps

#     def forward(self, x):
#         return F.adaptive_avg_pool2d(
#             x.clamp(min=self.eps).pow(self.p), output_size=1
#         ).pow(1.0 / self.p)


# class SubCenterArcCosine(nn.Module):
#     """Outputs raw cosine of features against K sub-centers per class.
#        Margin & scale applied in the LOSS, not here. Range: [-1, 1]."""
#     def __init__(self, in_features: int, num_classes: int, K: int = 3):
#         super().__init__()
#         self.K, self.num_classes = K, num_classes
#         self.weight = nn.Parameter(torch.empty(num_classes * K, in_features))
#         nn.init.xavier_uniform_(self.weight)

#     def forward(self, features):
#         f = F.normalize(features, dim=1)
#         w = F.normalize(self.weight, dim=1)
#         cos = f @ w.t()
#         cos = cos.view(-1, self.num_classes, self.K).max(dim=2).values
#         return cos


# # ---------------------------------------------------------------------
# # Model variants
# # ---------------------------------------------------------------------
# class EfficientSkin_B2_NS_GeM(nn.Module):
#     """B2 + GeM + BN-feature head + Linear classifier. Drop-in for CE+CutMix+SAM."""
#     def __init__(self, num_classes=31, pretrained_tag='efficientnet_b2',
#                  drop_rate=0.3, drop_path_rate=0.3, gem_p=3.0, pretrained=True):
#         super().__init__()
#         self.backbone = timm.create_model(
#             pretrained_tag, pretrained=pretrained, num_classes=0,
#             global_pool='', drop_rate=0.0, drop_path_rate=drop_path_rate,
#         )
#         feat_ch = self.backbone.num_features
#         self.pool = GeMPool(p=gem_p, learnable=True)
#         self.bn_feat = nn.BatchNorm1d(feat_ch)
#         self.dropout = nn.Dropout(drop_rate)
#         self.classifier = nn.Linear(feat_ch, num_classes)

#     def forward(self, x):
#         f = self.backbone.forward_features(x)
#         f = self.pool(f).flatten(1)
#         f = self.bn_feat(f)
#         return self.classifier(self.dropout(f))


# class EfficientSkin_B2_ArcFace(nn.Module):
#     """B2 + GeM + Sub-Center ArcFace. Returns raw cosine."""
#     def __init__(self, num_classes=31, pretrained_tag='efficientnet_b2',
#                  drop_rate=0.3, drop_path_rate=0.3, gem_p=3.0, arc_K=3,
#                  pretrained=True):
#         super().__init__()
#         self.backbone = timm.create_model(
#             pretrained_tag, pretrained=pretrained, num_classes=0,
#             global_pool='', drop_rate=0.0, drop_path_rate=drop_path_rate,
#         )
#         feat_ch = self.backbone.num_features
#         self.pool = GeMPool(p=gem_p, learnable=True)
#         self.bn_feat = nn.BatchNorm1d(feat_ch)
#         self.dropout = nn.Dropout(drop_rate)
#         self.classifier = SubCenterArcCosine(feat_ch, num_classes, K=arc_K)

#     def forward(self, x):
#         f = self.backbone.forward_features(x)
#         f = self.pool(f).flatten(1)
#         f = self.bn_feat(f)
#         return self.classifier(self.dropout(f))   # raw cosine


# class EfficientSkin_B2_ArcFace_AuxKD(nn.Module):
#     """B2 + GeM + Sub-Center ArcFace + auxiliary Linear head for self-KD.
#        Returns dict {'main': cosine, 'aux': aux_logits or None}."""
#     def __init__(self, num_classes=31, pretrained_tag='efficientnet_b2',
#                  drop_rate=0.3, drop_path_rate=0.3, gem_p=3.0, arc_K=3,
#                  aux_hidden=704, pretrained=True):
#         super().__init__()
#         self.backbone = timm.create_model(
#             pretrained_tag, pretrained=pretrained, num_classes=0,
#             global_pool='', drop_rate=0.0, drop_path_rate=drop_path_rate,
#         )
#         feat_ch = self.backbone.num_features
#         self.pool = GeMPool(p=gem_p, learnable=True)
#         self.bn_feat = nn.BatchNorm1d(feat_ch)
#         self.dropout = nn.Dropout(drop_rate)
#         self.main_head = SubCenterArcCosine(feat_ch, num_classes, K=arc_K)
#         self.aux_head = nn.Sequential(
#             nn.Linear(feat_ch, aux_hidden, bias=False),
#             nn.BatchNorm1d(aux_hidden),
#             nn.SiLU(inplace=True),
#             nn.Dropout(drop_rate),
#             nn.Linear(aux_hidden, num_classes),
#         )

#     def forward(self, x):
#         f = self.backbone.forward_features(x)
#         f = self.pool(f).flatten(1)
#         f = self.bn_feat(f)
#         f_d = self.dropout(f)
#         main = self.main_head(f_d)
#         if self.training:
#             aux = self.aux_head(f_d)
#             return {'main': main, 'aux': aux}
#         return {'main': main, 'aux': None}


# # ---------------------------------------------------------------------
# # ArcFace loss with CutMix support
# # ---------------------------------------------------------------------
# class ArcFaceLoss(nn.Module):
#     """Sub-Center ArcFace loss. Cutmix-aware (y_a/y_b/lam interface)."""
#     def __init__(self, s=30.0, m=0.35, label_smoothing=0.1, easy_margin=False):
#         super().__init__()
#         self.s, self.m, self.ls = s, m, label_smoothing
#         self.easy_margin = easy_margin
#         self.cos_m = math.cos(m)
#         self.sin_m = math.sin(m)
#         self.th    = math.cos(math.pi - m)
#         self.mm    = math.sin(math.pi - m) * m

#     def _apply_margin(self, cos, labels):
#         sin = torch.sqrt((1.0 - cos.pow(2)).clamp(min=1e-9))
#         cos_phi = cos * self.cos_m - sin * self.sin_m
#         if self.easy_margin:
#             cos_phi = torch.where(cos > 0, cos_phi, cos)
#         else:
#             cos_phi = torch.where(cos > self.th, cos_phi, cos - self.mm)
#         one_hot = torch.zeros_like(cos).scatter_(1, labels.view(-1, 1), 1.0)
#         return one_hot * cos_phi + (1.0 - one_hot) * cos

#     def forward(self, cos, y_a, y_b=None, lam=1.0):
#         if y_b is None or lam >= 1.0 - 1e-8:
#             return F.cross_entropy(
#                 self.s * self._apply_margin(cos, y_a), y_a,
#                 label_smoothing=self.ls
#             )
#         la = F.cross_entropy(self.s * self._apply_margin(cos, y_a), y_a,
#                              label_smoothing=self.ls)
#         lb = F.cross_entropy(self.s * self._apply_margin(cos, y_b), y_b,
#                              label_smoothing=self.ls)
#         return lam * la + (1.0 - lam) * lb


# def _ce_cutmix(logits, y_a, y_b=None, lam=1.0, label_smoothing=0.1):
#     if y_b is None or lam >= 1.0 - 1e-8:
#         return F.cross_entropy(logits, y_a, label_smoothing=label_smoothing)
#     return (lam * F.cross_entropy(logits, y_a, label_smoothing=label_smoothing)
#             + (1.0 - lam) * F.cross_entropy(logits, y_b, label_smoothing=label_smoothing))


# def _kd_loss(student, teacher, T=4.0):
#     s = F.log_softmax(student / T, dim=1)
#     t = F.softmax(teacher / T, dim=1).detach()
#     return F.kl_div(s, t, reduction='batchmean') * (T ** 2)


# # Identify Q1 variants
# _Q1_VARIANTS = {'b2_ns_gem', 'b2_arcface', 'b2_arcface_auxkd'}


# # ---------------------------------------------------------------------
# # Monkey-patch build_model: handle new variants, fall back to old for others
# # ---------------------------------------------------------------------
# _old_build_model = build_model

# def build_model(
#     model_name="efficientnet_b2",
#     model_variant="original",
#     drop_rate=0.0,
#     pretrained=True,
#     # ---- Q1 hyperparameters ----
#     q1_pretrained_tag='efficientnet_b2',
#     q1_drop_path_rate=0.3,
#     q1_gem_p=3.0,
#     q1_arc_K=3,
#     q1_aux_hidden=704,
#     **kwargs,
# ):
#     if model_variant in _Q1_VARIANTS:
#         # Effective dropout: if user passed drop_rate=0.0 (your STRONG default),
#         # use a head dropout of 0.3 since Q1 heads need some regularization.
#         effective_drop = drop_rate if drop_rate > 0 else 0.3

#         if model_variant == 'b2_ns_gem':
#             model = EfficientSkin_B2_NS_GeM(
#                 num_classes=num_classes, pretrained_tag=q1_pretrained_tag,
#                 drop_rate=effective_drop, drop_path_rate=q1_drop_path_rate,
#                 gem_p=q1_gem_p, pretrained=pretrained,
#             )
#         elif model_variant == 'b2_arcface':
#             model = EfficientSkin_B2_ArcFace(
#                 num_classes=num_classes, pretrained_tag=q1_pretrained_tag,
#                 drop_rate=effective_drop, drop_path_rate=q1_drop_path_rate,
#                 gem_p=q1_gem_p, arc_K=q1_arc_K, pretrained=pretrained,
#             )
#             model.uses_b2_q1_arcface = True
#         elif model_variant == 'b2_arcface_auxkd':
#             model = EfficientSkin_B2_ArcFace_AuxKD(
#                 num_classes=num_classes, pretrained_tag=q1_pretrained_tag,
#                 drop_rate=effective_drop, drop_path_rate=q1_drop_path_rate,
#                 gem_p=q1_gem_p, arc_K=q1_arc_K, aux_hidden=q1_aux_hidden,
#                 pretrained=pretrained,
#             )
#             model.uses_b2_q1_arcface = True
#             model.uses_b2_q1_auxkd = True
#         return model.to(device)

#     # All non-Q1 variants → original build_model
#     return _old_build_model(
#         model_name=model_name, model_variant=model_variant,
#         drop_rate=drop_rate, pretrained=pretrained, **kwargs
#     )


# # ---------------------------------------------------------------------
# # Monkey-patch forward_loss: handle Q1 variants, fall back for others
# # ---------------------------------------------------------------------
# _old_forward_loss = forward_loss

# def forward_loss(
#     model, images, labels, criterion,
#     labels_a=None, labels_b=None, lam=1.0,
#     use_amp=True, aux_weight=0.2,
#     coarse_weight=0.0, fine_to_coarse_tensor=None,
#     compact_weight=0.0, sep_weight=0.0,
#     # --- Q1 extras (read off `model`) ---
#     arc_s=30.0, arc_m=0.35, kd_T=4.0, kd_weight=0.5, aux_ce_weight=0.3,
#     **_,
# ):
#     amp_device = "cuda" if torch.cuda.is_available() else "cpu"
#     uses_q1_arc   = getattr(model, "uses_b2_q1_arcface", False)
#     uses_q1_auxkd = getattr(model, "uses_b2_q1_auxkd", False)

#     if not (uses_q1_arc or uses_q1_auxkd):
#         # Not a Q1 variant → defer to original
#         return _old_forward_loss(
#             model=model, images=images, labels=labels, criterion=criterion,
#             labels_a=labels_a, labels_b=labels_b, lam=lam,
#             use_amp=use_amp, aux_weight=aux_weight,
#             coarse_weight=coarse_weight, fine_to_coarse_tensor=fine_to_coarse_tensor,
#             compact_weight=compact_weight, sep_weight=sep_weight,
#         )

#     # Q1 path
#     ls = getattr(criterion, 'label_smoothing', 0.1)
#     arc_loss_fn = ArcFaceLoss(s=arc_s, m=arc_m, label_smoothing=ls)

#     with torch.amp.autocast(amp_device,
#                             enabled=use_amp and torch.cuda.is_available()):
#         out = model(images)

#         if uses_q1_auxkd:
#             # dict output {'main': cos, 'aux': logits}
#             main_cos = out['main']
#             aux_log  = out['aux']      # None in eval (training=False)

#             l_arc = arc_loss_fn(main_cos, labels_a if labels_a is not None else labels,
#                                 labels_b, lam)
#             if aux_log is None:
#                 loss = l_arc
#             else:
#                 l_aux = _ce_cutmix(aux_log,
#                                    labels_a if labels_a is not None else labels,
#                                    labels_b, lam, label_smoothing=ls)
#                 main_logits = arc_s * main_cos
#                 l_kd = 0.5 * (_kd_loss(main_logits, aux_log, T=kd_T)
#                               + _kd_loss(aux_log, main_logits, T=kd_T))
#                 loss = l_arc + aux_ce_weight * l_aux + kd_weight * l_kd

#             # outputs for accuracy: argmax of cosine == argmax of softmax(s*cos)
#             outputs = main_cos
#         else:
#             # b2_arcface: just raw cosine
#             l_arc = arc_loss_fn(out, labels_a if labels_a is not None else labels,
#                                 labels_b, lam)
#             loss = l_arc
#             outputs = out

#     return outputs, loss


# print("✓ Q1 module loaded. Variants available: b2_ns_gem, b2_arcface, b2_arcface_auxkd")
# print(f"  build_model and forward_loss monkey-patched (old versions saved).")


In [ ]:
# baseline_model = efficientnet_b2_original(
#     num_classes=num_classes,
#     pretrained=False,
#     drop_rate=0.0
# )

# upg_e384_model = efficientnet_b2_upg(
#     num_classes=num_classes,
#     pretrained=False,
#     drop_rate=0.0,
#     embedding_dim=384,
#     prototype_scale=12.0,
#     classifier_scale=1.0,
#     uncertainty_hidden=128,
#     use_gem=True
# )

# baseline_stats = count_params_flops(baseline_model)
# upg_e384_stats = count_params_flops(upg_e384_model)

# compare_df = pd.DataFrame([
#     {
#         "model": "Strong EfficientNet-B2",
#         "best_acc": 0.9014,
#         **baseline_stats
#     },
#     {
#         "model": "Strong EfficientNet-B2 + UPG e384",
#         "best_acc": 0.9044,
#         **upg_e384_stats
#     }
# ])

# compare_df["delta_acc"] = compare_df["best_acc"] - compare_df.loc[0, "best_acc"]
# compare_df["delta_params_m"] = compare_df["params_m"] - compare_df.loc[0, "params_m"]
# compare_df["delta_macs_g"] = compare_df["macs_g"] - compare_df.loc[0, "macs_g"]
# compare_df["params_increase_%"] = compare_df["delta_params_m"] / compare_df.loc[0, "params_m"] * 100
# compare_df["macs_increase_%"] = compare_df["delta_macs_g"] / compare_df.loc[0, "macs_g"] * 100

# display(compare_df)

In [ ]:
# teacher_path = "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best_upg/1/efficientnet-b2_strong_plus_upg_scalar_e384_ps12_best.pth"
# print(Path(teacher_path).exists())

# Analysis utilities

# Load best checkpoint for diagnosis

# Analyze train set

# Refined training dataset

In [ ]:
exclude_paths = []

if dataset_name == "skin31":
    train_sets = [
        FilteredImageFolder(skin31_train_dir, transform=orig_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=zoom_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=rot_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=bright_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=shear_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=vflip_tf, exclude_paths=exclude_paths),
        FilteredImageFolder(skin31_train_dir, transform=hflip_tf, exclude_paths=exclude_paths)
    ]

    train_dataset = ConcatDataset(train_sets)
    test_dataset = datasets.ImageFolder(skin31_test_dir, transform=to_tensor_norm)

    base_train_refined = FilteredImageFolder(skin31_train_dir, transform=None, exclude_paths=exclude_paths)
    base_targets = np.array(base_train_refined.targets)

elif dataset_name == "ham10000":
    train_sets = [
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=orig_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=zoom_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=rot_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=bright_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=shear_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=vflip_tf, exclude_paths=exclude_paths),
        HAM10000Dataset(ham_train_df, class_names=class_names, transform=hflip_tf, exclude_paths=exclude_paths)
    ]

    train_dataset = ConcatDataset(train_sets)
    test_dataset = HAM10000Dataset(ham_test_df, class_names=class_names, transform=to_tensor_norm)

    base_train_refined = HAM10000Dataset(ham_train_df, class_names=class_names, transform=None, exclude_paths=exclude_paths)
    base_targets = np.array(base_train_refined.targets)

else:
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")

class_counts = np.bincount(base_targets, minlength=num_classes)
class_weights = np.zeros_like(class_counts, dtype=np.float64)

nonzero_mask = class_counts > 0
class_weights[nonzero_mask] = 1.0 / class_counts[nonzero_mask]
sample_weights = class_weights[base_targets]
sample_weights = np.tile(sample_weights, len(train_sets))

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print("Dataset:", dataset_name)
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))
print("Num classes:", num_classes)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Retrain experiments

In [ ]:
# # =====================================================================
# # Q1 experiments — uses your existing run_experiment, STRONG config unchanged
# # =====================================================================
# Q1_STRONG = dict(
#     optimizer_name  = "adam",
#     scheduler_name  = "cosine",
#     lr_value        = 2.5e-4,
#     label_smoothing = 0.1,
#     use_cutmix      = True,
#     cutmix_alpha    = 1.0,
#     use_sam         = True,
#     sam_rho         = 0.1,
#     drop_rate       = 0.0,        # head will use 0.3 internally for Q1 variants
#     run_diagnostic  = False,
# )

# q1_experiments = [
#     # ── Step 1: GeM + BN-feat head (sanity ablation; no ArcFace yet) ───
#     {**Q1_STRONG,
#      "exp_name":      "Q1 - B2 + GeM head",
#      "model_variant": "b2_ns_gem"},

#     # ── Step 2: Sub-Center ArcFace ─────────────────────────────────────
#     {**Q1_STRONG,
#      "exp_name":      "Q1 - B2 + ArcFace (m0.35)",
#      "model_variant": "b2_arcface"},

#     # ── Step 3: PRIMARY — ArcFace + auxiliary self-KD head ─────────────
#     {**Q1_STRONG,
#      "exp_name":      "Q1 - B2 + ArcFace + AuxKD",
#      "model_variant": "b2_arcface_auxkd"},

#     # ── Step 4 (optional): same as Step 3 but with NoisyStudent pretrain.
#     # REQUIRES updating transforms to use the right normalization for
#     # tf_efficientnet_b2.ns_jft_in1k. See note at bottom of this cell.
#     # Uncomment ONLY after you've patched transforms.
#     {**Q1_STRONG,
#      "exp_name":      "Q1 - B2(NS) + ArcFace + AuxKD",
#      "model_variant": "b2_arcface_auxkd"},
# ]


# # Args allow-list for run_experiment (matches the existing call signature)
# _Q1_ALLOWED_ARGS = {
#     "exp_name", "optimizer_name", "scheduler_name", "lr_value",
#     "label_smoothing", "use_cutmix", "cutmix_alpha",
#     "use_sam", "sam_rho", "sam_adaptive", "drop_rate",
#     "aux_weight", "run_diagnostic", "baseline_checkpoint",
# }


# q1_results = []
# for exp in q1_experiments:
#     try:
#         result = run_experiment(
#             **{k: v for k, v in exp.items() if k in _Q1_ALLOWED_ARGS},
#             model_name="efficientnet_b2",
#             model_variant=exp["model_variant"],
#         )
#     except Exception as e:
#         result = {
#             "experiment":    exp["exp_name"],
#             "model_variant": exp["model_variant"],
#             "best_acc":      None,
#             "best_epoch":    None,
#             "status":        f"failed: {repr(e)}",
#         }
#         print(f"[{exp['exp_name']}] FAILED → {repr(e)}")
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

#     q1_results.append(result)


# # ─── save & display ─────────────────────────────────────────────────
# q1_df = (
#     pd.DataFrame(q1_results)
#     .sort_values("best_acc", ascending=False, na_position="last")
#     .reset_index(drop=True)
# )
# q1_df.to_csv(work_dir / "q1_summary.csv", index=False)
# display(q1_df[["experiment", "model_variant", "best_acc", "best_epoch", "status"]])


# # ─────────────────────────────────────────────────────────────────────
# # NOTE on NoisyStudent pretrained weights (optional Step 4)
# # ─────────────────────────────────────────────────────────────────────
# # If you want to swap `efficientnet_b2` → `tf_efficientnet_b2.ns_jft_in1k`:
# #
# # 1. Get the right normalization for this checkpoint:
# #
# #       _probe = timm.create_model('tf_efficientnet_b2.ns_jft_in1k',
# #                                   pretrained=False)
# #       cfg = _probe.pretrained_cfg
# #       print("Expected mean/std:", cfg['mean'], cfg['std'])
# #
# # 2. If mean/std DIFFER from your imagenet_mean/imagenet_std in cell 16,
# #    rebuild ALL transforms (orig_tf, zoom_tf, rot_tf, ...) with the new
# #    values. WRONG normalization will give worse results than baseline.
# #
# # 3. Pass via build_model_q1 by editing the dispatch:
# #       q1_pretrained_tag='tf_efficientnet_b2.ns_jft_in1k'
# #    (set in the build_model wrapper or pass kwarg through run_experiment).


In [ ]:
# STRONG = dict(
#     optimizer_name  = "adam",
#     scheduler_name  = "cosine",
#     lr_value        = 2.5e-4,
#     label_smoothing = 0.1,
#     use_cutmix      = True,
#     cutmix_alpha    = 1.0,
#     use_sam         = True,
#     sam_rho         = 0.1,
#     drop_rate       = 0.0,
#     run_diagnostic  = False,
# )
 
# msda_v2_experiments = [
#     # ── Ablation Row 6 — TADA + Proto (PRIMARY — fixes conv_head bypass)
#     {**STRONG,
#      "exp_name": "MSDA v2 - TADA+Proto",
#      "model_variant": "msda_tada_proto",
#      "aux_weight": 0.2,
#      "proto_temperature": 0.07},
 
#     # ── Ablation Row 7 — TADA + GeM (pooling change only, no aux head)
#     {**STRONG,
#      "exp_name": "MSDA v2 - TADA+GeM",
#      "model_variant": "msda_tada_gem"},
 
#     # ── Ablation Row 8 — TADA + GeM + Proto (full v2 combo)
#     {**STRONG,
#      "exp_name": "MSDA v2 - TADA+Proto+GeM",
#      "model_variant": "msda_tada_proto_gem",
#      "aux_weight": 0.2,
#      "proto_temperature": 0.07},
 
#     # ── Ablation Row 9 — Fixed MSDA, D=128 (isolates backbone fix only)
#     {**STRONG,
#      "exp_name": "MSDA v2 - Full Fixed D128",
#      "model_variant": "msda_full_fixed",
#      "bifpn_dim": 128,
#      "aux_weight": 0.2,
#      "proto_temperature": 0.07,
#      "msda_transformer_depth": 2,
#      "msda_freeze_stages": 2},
 
#     # ── Ablation Row 10 — Fixed MSDA, D=256 (exploits richer P5)
#     {**STRONG,
#      "exp_name": "MSDA v2 - Full Fixed D256",
#      "model_variant": "msda_full_fixed",
#      "bifpn_dim": 256,
#      "aux_weight": 0.2,
#      "proto_temperature": 0.07,
#      "msda_transformer_depth": 2,
#      "msda_freeze_stages": 2},
# ]
 
# # ─── runner (same pattern as existing msda_experiments cell) ──────────────
# all_msda_v2_results = []
 
# for exp in msda_v2_experiments:
#     try:
#         result = run_experiment(
#             **{k: v for k, v in exp.items()
#                if k in [
#                    "exp_name", "optimizer_name", "scheduler_name", "lr_value",
#                    "label_smoothing", "use_cutmix", "cutmix_alpha",
#                    "use_sam", "sam_rho", "sam_adaptive", "drop_rate",
#                    "aux_weight", "run_diagnostic", "baseline_checkpoint",
#                    "bifpn_dim", "proto_temperature",
#                    "msda_transformer_depth", "msda_freeze_stages",
#                ]},
#             model_name="efficientnet_b2",
#             model_variant=exp["model_variant"],
#         )
#     except Exception as e:
#         result = {
#             "experiment": exp["exp_name"],
#             "model_variant": exp["model_variant"],
#             "best_acc": None,
#             "best_epoch": None,
#             "status": f"failed: {repr(e)}",
#         }
#         print(f"[{exp['exp_name']}] FAILED → {repr(e)}")
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()
 
#     all_msda_v2_results.append(result)
 
# # ─── save & display ───────────────────────────────────────────────────────
# v2_df = (
#     pd.DataFrame(all_msda_v2_results)
#     .sort_values("best_acc", ascending=False, na_position="last")
#     .reset_index(drop=True)
# )
# v2_df.to_csv(work_dir / "msda_v2_ablation_summary.csv", index=False)
# display(v2_df[["experiment", "model_variant", "best_acc", "best_epoch", "status"]])

In [ ]:
# # ────────────────────────────────────────────────────────────────────────────
# # MSDA Ablation — chạy độc lập hoặc merge vào `experiments` list ở cell trên
# # Training setting: LS0.1 + CutMix1.0 + SAM0.1 + lr=2.5e-4 (best known setup)
# # ────────────────────────────────────────────────────────────────────────────

# STRONG = dict(
#     optimizer_name  = "adam",
#     scheduler_name  = "cosine",
#     lr_value        = 2.5e-4,
#     label_smoothing = 0.1,
#     use_cutmix      = True,
#     cutmix_alpha    = 1.0,
#     use_sam         = True,
#     sam_rho         = 0.1,
#     drop_rate       = 0.0,
#     run_diagnostic  = False,
# )

# msda_experiments = [
#     # Ablation Row 1 — attention module only
#     {**STRONG, "exp_name": "MSDA - TADA only",   "model_variant": "msda_tada"},
#     # Ablation Row 2 — multi-scale neck only
#     {**STRONG, "exp_name": "MSDA - BiFPN only",  "model_variant": "msda_bifpn",  "bifpn_dim": 128},
#     # Ablation Row 3 — transformer global context only
#     {**STRONG, "exp_name": "MSDA - LWT only",    "model_variant": "msda_lwt",    "bifpn_dim": 128},
#     # Ablation Row 4 — prototype contrastive head only
#     {**STRONG, "exp_name": "MSDA - Proto only",  "model_variant": "msda_proto",  "aux_weight": 0.2, "proto_temperature": 0.07},
#     # Ablation Row 5 — full model
#     {**STRONG, "exp_name": "MSDA - Full",        "model_variant": "msda_full",
#      "bifpn_dim": 128, "aux_weight": 0.2, "proto_temperature": 0.07,
#      "msda_transformer_depth": 2, "msda_freeze_stages": 2},
#     # Proto weight tuning
#     {**STRONG, "exp_name": "MSDA - Full proto_w0.1", "model_variant": "msda_full",
#      "bifpn_dim": 128, "aux_weight": 0.1, "proto_temperature": 0.07,
#      "msda_transformer_depth": 2, "msda_freeze_stages": 2},
#     {**STRONG, "exp_name": "MSDA - Full proto_w0.3", "model_variant": "msda_full",
#      "bifpn_dim": 128, "aux_weight": 0.3, "proto_temperature": 0.07,
#      "msda_transformer_depth": 2, "msda_freeze_stages": 2},
# ]

# all_msda_results = []
# for exp in msda_experiments:
#     try:
#         result = run_experiment(
#             **{k: v for k, v in exp.items()
#                if k in [
#                    "exp_name","optimizer_name","scheduler_name","lr_value",
#                    "label_smoothing","use_cutmix","cutmix_alpha","use_sam","sam_rho",
#                    "sam_adaptive","drop_rate","aux_weight","run_diagnostic",
#                    "baseline_checkpoint","bifpn_dim","proto_temperature",
#                    "msda_transformer_depth","msda_freeze_stages",
#                ]},
#             model_name="efficientnet_b2",
#             model_variant=exp["model_variant"],
#         )
#     except Exception as e:
#         result = {"experiment": exp["exp_name"], "model_variant": exp["model_variant"],
#                   "best_acc": None, "status": f"failed: {repr(e)}"}
#         print(f"[{exp['exp_name']}] FAILED -> {repr(e)}")
#         gc.collect()
#         torch.cuda.empty_cache() if torch.cuda.is_available() else None
#     all_msda_results.append(result)

# msda_df = pd.DataFrame(all_msda_results).sort_values("best_acc", ascending=False, na_position="last")
# msda_df.to_csv(work_dir / "msda_ablation_summary.csv", index=False)
# display(msda_df[["experiment","model_variant","best_acc","best_epoch","status"]])


In [ ]:

experiments = [    
    # {
    #     "exp_name": "Adam + cosine",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine"
    # },
    # {
    #     "exp_name": "Adam + no scheduler",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "none"
    # },
    # {
    #     "exp_name": "AdamW + no scheduler",
    #     "optimizer_name": "adamw",
    #     "scheduler_name": "none"
    # },
    # {
    #     "exp_name": "AdamW + cosine",
    #     "optimizer_name": "adamw",
    #     "scheduler_name": "cosine"
    # },
    # {
    #     "exp_name": "Adam + cosine + LS0.1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + CutMix1.0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.0,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.03",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.03
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.05",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.05
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + SAM0.1 + lr2.5e-4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.2,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.3",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.3,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Adam + cosine + Dropout0.4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": lr,
    #     "drop_rate": 0.4,
    #     "use_sam": False
    # },
    # {
    #     "exp_name": "Baseline + LS0.1 + CutMix1.0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0
    # },
    # {
    #     "exp_name": "Baseline + LS0.1 + CutMix1.0 + SAM0.1 + lr2.5e-4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "B0 baseline",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": False,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "B0 + SAM0.1",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "B0 + SAM0.1 + lr2.5e-4",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "gem",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + late ECA",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "late_eca",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ECA",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_eca",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,        
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ECA + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_eca_gem",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + GeM",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_gem",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + ArcFace",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_arcface",
    #     "fusion_channels": 512,
    #     "embedding_dim": 256,
    #     "arcface_s": 30.0,
    #     "arcface_m": 0.3,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + Gated Fusion",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_gated",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Strong B2 + Dual Scale + Aux C4",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_aux",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "aux_weight": 0.2
    # },
    # {
    #     "exp_name": "HAM10000 Plain Baseline",
    #     "model_name": "efficientnet_b2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "cutmix_alpha": 0.0,
    #     "lr_value": 5e-4,
    #     "use_sam": False,
    #     "sam_rho": 0.0
    # },
    # {
    #     "exp_name": "HAM10000 Plain Baseline + Dual Scale",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.0,
    #     "use_cutmix": False,
    #     "cutmix_alpha": 0.0,
    #     "lr_value": 5e-4,
    #     "use_sam": False,
    #     "sam_rho": 0.0
    # },
    # {
    #     "exp_name": "HAM10000 Strong Baseline",
    #     "model_name": "efficientnet_b2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "HAM10000 Strong Baseline + Dual Scale",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale",
    #     "fusion_channels": 512,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "HAM10000 Plain Baseline + Dual Scale ArcFace clean",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_scale_arcface",
    #     "fusion_channels": 512,
    #     "embedding_dim": 256,
    #     "arcface_s": 24.0,
    #     "arcface_m": 0.15,
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    # },
    # {
    #     "exp_name": "HAM10000 Plain B2 + ArcFace",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "arcface",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "embedding_dim": 256,
    #     "arcface_s": 24.0,
    #     "arcface_m": 0.15,
    #     "lr_value": 5e-4,
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Regional Head",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "regional",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "embedding_dim": 256,
    #     "region_grid": 2,
    #     "use_gem": True,
    #     "attn_hidden": 128,
    #     "lr_value": 5e-4
    # },
    # {
    #     "exp_name": "Skin31 Strong B2 + Regional Head",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "regional",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "embedding_dim": 256,
    #     "region_grid": 2,
    #     "use_gem": True,
    #     "attn_hidden": 128,
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Spatial Refine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "spatial_refine",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "refine_kernel_size": 7
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Compact Head",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "compact_head",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "embedding_dim": 256,
    #     "use_gem": True,
    #     "drop_rate": 0.2,
    #     "use_cosine_classifier": False,
    #     "cosine_s": 24.0
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Compact Head + Cosine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "compact_head",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 5e-4,
    #     "embedding_dim": 256,
    #     "use_gem": True,
    #     "drop_rate": 0.2,
    #     "use_cosine_classifier": True,
    #     "cosine_s": 24.0
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Dual Evidence",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_evidence",
    #     "lr_value": 5e-4,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 256,
    #     "region_grid": 3,
    #     "top_k": 3,
    #     "use_gem": True,
    #     "score_hidden": 128
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Dual Evidence k1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_evidence",
    #     "lr_value": 5e-4,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 256,
    #     "region_grid": 3,
    #     "top_k": 1,
    #     "use_gem": True,
    #     "score_hidden": 128
    # },
    # {
    #     "exp_name": "Skin31 Plain B2 + Dual Evidence k5",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "dual_evidence",
    #     "lr_value": 5e-4,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 256,
    #     "region_grid": 3,
    #     "top_k": 5,
    #     "use_gem": True,
    #     "score_hidden": 128
    # },
    # {
    #     "exp_name": "Skin31 Strong B2 + C4 Refine",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "c4_refine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "lr_value": 2.5e-4,
    #     "use_sam": True,
    #     "sam_rho": 0.1
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + Mamba Refine d96",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "mamba_refine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "mamba_dim": 96,
    #     "d_state": 16,
    #     "d_conv": 3,
    #     "expand": 2,
    #     "init_alpha": 0.1,
    #     "use_external_mamba": False
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + Mamba Refine d64 a0.05",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "mamba_refine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "mamba_dim": 64,
    #     "d_state": 8,
    #     "d_conv": 3,
    #     "expand": 1,
    #     "init_alpha": 0.05,
    #     "use_external_mamba": False
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Head",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "embedding_dim": 256,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar ps16",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 256,
    #     "prototype_scale": 16.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps12",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e512 ps12",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 512,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 uh256 ps12",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 256,
    #     "use_gem": True
    # },
    #     {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps10",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 10.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG Scalar e384 ps12 cs0.8",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 0.8,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG e384 ps12 SelfDistill a0.3 T4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "use_kd": True,
    #     "kd_teacher_path": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best_upg/1/efficientnet-b2_strong_plus_upg_scalar_e384_ps12_best.pth",
    #     "kd_alpha": 0.3,
    #     "kd_temperature": 4.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + UPG e384 ps12 SelfDistill a0.5 T4",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "use_kd": True,
    #     "kd_teacher_path": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best_upg/1/efficientnet-b2_strong_plus_upg_scalar_e384_ps12_best.pth",
    #     "kd_alpha": 0.5,
    #     "kd_temperature": 4.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.05",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 0.5,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.05,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.1",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 0.5,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.1,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 0.5,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.2,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2 rs1.0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 1.0,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.2,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps10 cw0.2 rs0.5",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 10.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 0.5,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.2,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0
    # },
    # {
    #     "exp_name": "EfficientNet-B2 Strong + HierProto-UPG e384 ps12 cw0.2 rs1.0 gateT0.5",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "lr_value": 2.5e-4,
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "upg",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "drop_rate": 0.0,
    #     "embedding_dim": 384,
    #     "prototype_scale": 12.0,
    #     "classifier_scale": 1.0,
    #     "uncertainty_hidden": 128,
    #     "use_gem": True,
    #     "num_coarse_classes": num_coarse_classes,
    #     "fine_to_coarse": fine_to_coarse_list,
    #     "residual_scale": 1.0,
    #     "coarse_scale": 12.0,
    #     "coarse_weight": 0.2,
    #     "compact_weight": 0.0,
    #     "sep_weight": 0.0,
    #     "gate_temperature": 0.5,
    #     "run_diagnostic": True,
    #     "baseline_checkpoint": "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/baseline-ls-cutmix-sam/1/baseline_plus_ls0.1_plus_cutmix1.0_plus_sam0.1_plus_lr2.5e-4_best.pth"
    # },
    # {
    #     "exp_name": "WAGF EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "wagf",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "lr_value": 2.5e-4,
    #     "drop_rate": 0.0,
    #     "run_diagnostic": False
    # },
    # {
    #     "exp_name": "ECA EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "eca",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "lr_value": 2.5e-4,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "CBAM EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "cbam",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "lr_value": 2.5e-4,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "MSSF EfficientNet-B2 + LS0.1 + CutMix1.0 + SAM0.1",
    #     "model_name": "efficientnet_b2",
    #     "model_variant": "mssf",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "lr_value": 2.5e-4,
    #     "drop_rate": 0.0
    # },
    # {
    #     "exp_name": "B0 + LS0.1 + CutMix1.0 + SAM0.1",
    #     "model_name": "efficientnet_b0",
    #     "optimizer_name": "adam",
    #     "scheduler_name": "cosine",
    #     "label_smoothing": 0.1,
    #     "use_cutmix": True,
    #     "cutmix_alpha": 1.0,
    #     "use_sam": True,
    #     "sam_rho": 0.1,
    #     "lr_value": 2.5e-4,
    #     "drop_rate": 0.0,
    # }
]

In [ ]:
# all_results = []

# for exp in experiments:
#     try:
#         result = run_experiment(
#             exp_name=exp["exp_name"],
#             optimizer_name=exp["optimizer_name"],
#             scheduler_name=exp["scheduler_name"],
#             lr_value=exp.get("lr_value", None),
#             model_name=exp.get("model_name", "efficientnet_b2"),
#             model_variant=exp.get("model_variant", "original"),
#             label_smoothing=exp.get("label_smoothing", 0.0),
#             use_cutmix=exp.get("use_cutmix", False),
#             cutmix_alpha=exp.get("cutmix_alpha", 1.0),
#             use_sam=exp.get("use_sam", False),
#             sam_rho=exp.get("sam_rho", 0.05),
#             sam_adaptive=exp.get("sam_adaptive", False),
#             drop_rate=exp.get("drop_rate", 0.0),
#             fusion_channels=exp.get("fusion_channels", 512),
#             embedding_dim=exp.get("embedding_dim", 256),
#             arcface_s=exp.get("arcface_s", 30.0),
#             arcface_m=exp.get("arcface_m", 0.3),
#             aux_weight=exp.get("aux_weight", 0.2),
#             region_grid=exp.get("region_grid", 2),
#             top_k=exp.get("top_k", 3),
#             use_gem=exp.get("use_gem", True),
#             attn_hidden=exp.get("attn_hidden", 128),
#             score_hidden=exp.get("score_hidden", 128),
#             refine_kernel_size=exp.get("refine_kernel_size", 7),
#             use_cosine_classifier=exp.get("use_cosine_classifier", False),
#             cosine_s=exp.get("cosine_s", 24.0),
#             mamba_dim=exp.get("mamba_dim", 96),
#             d_state=exp.get("d_state", 16),
#             d_conv=exp.get("d_conv", 3),
#             expand=exp.get("expand", 2),
#             init_alpha=exp.get("init_alpha", 0.1),
#             use_external_mamba=exp.get("use_external_mamba", False),
#             prototype_scale=exp.get("prototype_scale", 12.0),
#             classifier_scale=exp.get("classifier_scale", 1.0),
#             uncertainty_hidden=exp.get("uncertainty_hidden", 128),
#             use_kd=exp.get("use_kd", False),
#             kd_teacher_path=exp.get("kd_teacher_path", None),
#             kd_alpha=exp.get("kd_alpha", 0.3),
#             kd_temperature=exp.get("kd_temperature", 4.0),
#             num_coarse_classes=exp.get("num_coarse_classes", 0),
#             fine_to_coarse=exp.get("fine_to_coarse", None),
#             residual_scale=exp.get("residual_scale", 0.5),
#             coarse_scale=exp.get("coarse_scale", 12.0),
#             coarse_weight=exp.get("coarse_weight", 0.0),
#             compact_weight=exp.get("compact_weight", 0.0),
#             sep_weight=exp.get("sep_weight", 0.0),
#             gate_temperature=exp.get("gate_temperature", 1.0),
#             run_diagnostic=exp.get("run_diagnostic", False),
#             baseline_checkpoint=exp.get("baseline_checkpoint", None),
#             bifpn_dim=exp.get("bifpn_dim", 128),
#             proto_temperature=exp.get("proto_temperature", 0.07),
#             msda_transformer_depth=exp.get("msda_transformer_depth", 2),
#             msda_freeze_stages=exp.get("msda_freeze_stages", 2),
#         )
#     except Exception as e:
#         result = {
#             "experiment": exp["exp_name"],
#             "model_name": exp.get("model_name", "efficientnet_b2"),
#             "model_variant": exp.get("model_variant", "original"),
#             "optimizer": exp["optimizer_name"],
#             "scheduler": exp["scheduler_name"],
#             "lr": float(exp.get("lr_value", lr)),
#             "label_smoothing": float(exp.get("label_smoothing", 0.0)),
#             "use_cutmix": bool(exp.get("use_cutmix", False)),
#             "cutmix_alpha": float(exp.get("cutmix_alpha", 1.0)),
#             "use_sam": bool(exp.get("use_sam", False)),
#             "sam_rho": float(exp.get("sam_rho", 0.05)),
#             "sam_adaptive": bool(exp.get("sam_adaptive", False)),
#             "drop_rate": float(exp.get("drop_rate", 0.0)),
#             "fusion_channels": int(exp.get("fusion_channels", 512)),
#             "embedding_dim": int(exp.get("embedding_dim", 256)),
#             "arcface_s": float(exp.get("arcface_s", 30.0)),
#             "arcface_m": float(exp.get("arcface_m", 0.3)),
#             "aux_weight": float(exp.get("aux_weight", 0.2)),
#             "region_grid": int(exp.get("region_grid", 2)),
#             "top_k": int(exp.get("top_k", 3)),
#             "use_gem": bool(exp.get("use_gem", True)),
#             "attn_hidden": int(exp.get("attn_hidden", 128)),
#             "score_hidden": int(exp.get("score_hidden", 128)),
#             "refine_kernel_size": int(exp.get("refine_kernel_size", 7)),
#             "use_cosine_classifier": bool(exp.get("use_cosine_classifier", False)),
#             "cosine_s": float(exp.get("cosine_s", 24.0)),
#             "mamba_dim": int(exp.get("mamba_dim", 96)),
#             "d_state": int(exp.get("d_state", 16)),
#             "d_conv": int(exp.get("d_conv", 3)),
#             "expand": int(exp.get("expand", 2)),
#             "init_alpha": float(exp.get("init_alpha", 0.1)),
#             "use_external_mamba": bool(exp.get("use_external_mamba", False)),
#             "prototype_scale": float(exp.get("prototype_scale", 12.0)),
#             "classifier_scale": float(exp.get("classifier_scale", 1.0)),
#             "uncertainty_hidden": int(exp.get("uncertainty_hidden", 128)),
#             "use_kd": bool(exp.get("use_kd", False)),
#             "kd_teacher_path": exp.get("kd_teacher_path", None),
#             "kd_alpha": float(exp.get("kd_alpha", 0.3)),
#             "kd_temperature": float(exp.get("kd_temperature", 4.0)),
#             "num_coarse_classes": int(exp.get("num_coarse_classes") or 0),
#             "residual_scale": float(exp.get("residual_scale", 0.5)),
#             "coarse_scale": float(exp.get("coarse_scale", 12.0)),
#             "coarse_weight": float(exp.get("coarse_weight", 0.0)),
#             "compact_weight": float(exp.get("compact_weight", 0.0)),
#             "sep_weight": float(exp.get("sep_weight", 0.0)),
#             "gate_temperature": float(exp.get("gate_temperature", 1.0)),
#             "run_diagnostic": bool(exp.get("run_diagnostic", True)),
#             "baseline_checkpoint": exp.get("baseline_checkpoint", None),
#             "best_acc": None,
#             "best_epoch": None,
#             "save_path": None,
#             "status": f"failed: {repr(e)}"
#         }
#         print(f"[{exp['exp_name']}] FAILED -> {repr(e)}")
#         gc.collect()
#         if torch.cuda.is_available():
#             torch.cuda.empty_cache()

#     all_results.append(result)
#     summary_df, summary_csv_path, summary_json_path = save_summary_files(all_results)

# summary_df = pd.DataFrame(all_results).sort_values(
#     by=["best_acc"],
#     ascending=False,
#     na_position="last"
# ).reset_index(drop=True)

# summary_df.to_csv(work_dir / "experiment_summary_sorted.csv", index=False)

# print("\nFinal summary:")
# display(summary_df)

# all_history_list = []
# for exp in experiments:
#     exp_slug = slugify_exp_name(exp["exp_name"])
#     csv_path = history_dir / f"{exp_slug}_history.csv"
#     if csv_path.exists():
#         df_exp = pd.read_csv(csv_path)
#         all_history_list.append(df_exp)

# if len(all_history_list) > 0:
#     all_history = pd.concat(all_history_list, ignore_index=True)
#     all_history.to_csv(work_dir / "all_history.csv", index=False)
#     display(all_history)

#     plt.figure(figsize=(12, 5))
#     for exp_name in all_history["experiment"].unique():
#         df_exp = all_history[all_history["experiment"] == exp_name]
#         plt.plot(df_exp["epoch"], df_exp["test_acc"], label=exp_name)
#     plt.legend()
#     plt.xlabel("epoch")
#     plt.ylabel("test_acc")
#     plt.title("Test Accuracy")
#     plt.show()

#     plt.figure(figsize=(12, 5))
#     for exp_name in all_history["experiment"].unique():
#         df_exp = all_history[all_history["experiment"] == exp_name]
#         plt.plot(df_exp["epoch"], df_exp["train_acc"], label=exp_name)
#     plt.legend()
#     plt.xlabel("epoch")
#     plt.ylabel("train_acc")
#     plt.title("Train Accuracy")
#     plt.show()

#     plt.figure(figsize=(12, 5))
#     for exp_name in all_history["experiment"].unique():
#         df_exp = all_history[all_history["experiment"] == exp_name]
#         plt.plot(df_exp["epoch"], df_exp["test_loss"], label=exp_name)
#     plt.legend()
#     plt.xlabel("epoch")
#     plt.ylabel("test_loss")
#     plt.title("Test Loss")
#     plt.show()

#     plt.figure(figsize=(12, 5))
#     for exp_name in all_history["experiment"].unique():
#         df_exp = all_history[all_history["experiment"] == exp_name]
#         plt.plot(df_exp["epoch"], df_exp["train_loss"], label=exp_name)
#     plt.legend()
#     plt.xlabel("epoch")
#     plt.ylabel("train_loss")
#     plt.title("Train Loss")
#     plt.show()
# else:
#     print("No history files found.")

In [ ]:
TEACHER_PATH = "/kaggle/input/models/thanhkhil/efficientnet-b2-skin31/pytorch/best-dualscale/1/strong_b2_plus_dual_scale_best.pth"
 
# kd_result = run_experiment(
#     exp_name="B0 + KD(DualScale) + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",         # STUDENT
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     # ── KD ──────────────────────────────────────────
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",       # teacher kiến trúc
#     kd_teacher_fusion_channels=512,        # SỬA nếu teacher train với fc khác (256?)
#     kd_alpha=0.5,                          # 0.5 CE + 0.5 KD (gap nhỏ → cân bằng)
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )
# kd_result_07 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.7 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.7,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )

# kd_result_03 = run_experiment(
#     exp_name="B0 + KD(DualScale) a0.3 + LS0.1 + CutMix1.0 + SAM0.1",
#     model_name="efficientnet_b0",
#     model_variant="original",
#     optimizer_name="adam",
#     scheduler_name="cosine",
#     label_smoothing=0.1,
#     use_cutmix=True,
#     cutmix_alpha=1.0,
#     use_sam=True,
#     sam_rho=0.1,
#     lr_value=2.5e-4,
#     drop_rate=0.0,
#     use_kd=True,
#     kd_teacher_path=TEACHER_PATH,
#     kd_teacher_variant="dual_scale",
#     kd_teacher_fusion_channels=512,
#     kd_alpha=0.3,                          # ← đổi
#     kd_temperature=4.0,
#     run_diagnostic=False,
# )

kd_result_06 = run_experiment(
    exp_name="B0 + KD(DualScale) a0.6 + LS0.1 + CutMix1.0 + SAM0.1",
    model_name="efficientnet_b0",
    model_variant="original",
    optimizer_name="adam",
    scheduler_name="cosine",
    label_smoothing=0.1,
    use_cutmix=True,
    cutmix_alpha=1.0,
    use_sam=True,
    sam_rho=0.1,
    lr_value=2.5e-4,
    drop_rate=0.0,
    use_kd=True,
    kd_teacher_path=TEACHER_PATH,
    kd_teacher_variant="dual_scale",
    kd_teacher_fusion_channels=512,
    kd_alpha=0.6,                          # ← đổi
    kd_temperature=4.0,
    run_diagnostic=False,
)

kd_result_08 = run_experiment(
    exp_name="B0 + KD(DualScale) a0.8 + LS0.1 + CutMix1.0 + SAM0.1",
    model_name="efficientnet_b0",
    model_variant="original",
    optimizer_name="adam",
    scheduler_name="cosine",
    label_smoothing=0.1,
    use_cutmix=True,
    cutmix_alpha=1.0,
    use_sam=True,
    sam_rho=0.1,
    lr_value=2.5e-4,
    drop_rate=0.0,
    use_kd=True,
    kd_teacher_path=TEACHER_PATH,
    kd_teacher_variant="dual_scale",
    kd_teacher_fusion_channels=512,
    kd_alpha=0.8,                          # ← đổi
    kd_temperature=4.0,
    run_diagnostic=False,
)